In [3]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from profis.utils.modelinit import initialize_model
from profis.gen.dataset import SMILESDataset, SELFIESDataset, DeepSMILESDataset
from profis.utils.vectorizer import SMILESVectorizer, SELFIESVectorizer, DeepSMILESVectorizer
import configparser

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load model
config_path = 'config/config.yaml'
config = configparser.ConfigParser()
config.read(config_path)
big_model = initialize_model(config_path, device=device)
big_model.load_state_dict(torch.load(config_path.replace('hyperparameters.ini', 'model.pt'), map_location=device))

KeyError: 'RUN'

In [2]:
# prepare dataloaders
dataloader_workers = 3
batch_size = 128

fp_len = config.getint('model', 'fp_len')
fp_type = 'KRFP' if fp_len == 4860 else 'ECFP'
out_encoding = config.get('model', 'out_encoding')

# load data
if fp_type == 'ECFP':
    df = pd.read_parquet('data/RNN_dataset_ECFP_val_10.parquet')
elif fp_type == 'KRFP':
    df = pd.read_csv('data/RNN_dataset_KRFP_val_10.parquet')
else:
    raise ValueError('Invalid fingerprint type (must be ECFP or KRFP)')

if out_encoding == "selfies":
    vectorizer = SELFIESVectorizer(pad_to_len=128)
    dataset = SELFIESDataset(df, vectorizer, fp_len)

elif out_encoding == "smiles":
    vectorizer = SMILESVectorizer(pad_to_len=128)
    dataset = SMILESDataset(df, vectorizer, fp_len)

elif out_encoding == "deepsmiles":
    vectorizer = DeepSMILESVectorizer(pad_to_len=128)
    dataset = DeepSMILESDataset(df, vectorizer, fp_len)
else:
    raise ValueError(
        "Invalid output encoding (must be selfies, smiles or deepsmiles)"
    )

loader = DataLoader(
    dataset,
    shuffle=True,
    batch_size=batch_size,
    drop_last=True,
    num_workers=dataloader_workers,
)

NameError: name 'config' is not defined

In [ ]:
from profis.pred.pred import stochastic_decoder
from profis.gen.train import get_scores
import rdkit.Chem as Chem

mean_validity = 0
mean_qed = 0
mean_fp_recon = 0
preds = []
with torch.no_grad():
    for X, y in loader:
        y_hat = big_model(X.to(device))
        preds.append(y_hat.detach().cpu().numpy())
        qed, fp_recon, validity = get_scores(y_hat, y, vectorizer)
        mean_qed += qed
        mean_fp_recon += fp_recon
        mean_validity += validity
    mean_qed /= len(loader)
    mean_fp_recon /= len(loader)
    mean_validity /= len(loader)
    
print(f"Mean QED: {mean_qed}")
print(f"Mean FP reconstruction error: {mean_fp_recon}")
print(f"Mean validity: {mean_validity}")

decoded_preds = [stochastic_decoder(pred, vectorizer) for pred in preds]
mols = [Chem.MolFromSmiles(smiles) for smiles in decoded_preds]
valid = [mol is not None for mol in mols]
print(f"Final validity: {len(valid) / len(decoded_preds)}")